# ORIENT'IA — Analyse et entraînement du modèle de Machine Learning

Ce notebook rejoue, cellule par cellule, le pipeline implémenté sous `ml/` : génération des données synthétiques, analyse exploratoire, entraînement et comparaison de trois approches, évaluation sur un split de validation puis sur l'enquête réelle. Exécuter les cellules dans l'ordre depuis la racine du dépôt (kernel Python de `venv/`).

Le détail méthodologique (hypothèses, biais, contrôles) est documenté dans le code (`ml/synthetic.py`, `ml/survey.py`) et dans `ml/artifacts/evaluation_report.md` — ce notebook en donne une vue exécutable, pas une redite.

## 1. Génération des données synthétiques (jeu d'entraînement)

In [ ]:
import sys
sys.path.insert(0, '..')

from ml.synthetic import generate_dataset
from ml.domaines import DOMAINE_IDS

rows = generate_dataset(n_per_class=150, seed=42)
print(f"{len(rows)} profils générés pour {len(DOMAINE_IDS)} domaines d'orientation")
rows[0]

## 2. Analyse exploratoire

Vérification de l'équilibre des classes, du taux d'étiquettes bruitées introduites volontairement (voir la docstring de `ml.synthetic`), et de la fréquence des tags générés.

In [ ]:
from collections import Counter

label_counts = Counter(r['domaine_recommande'] for r in rows)
print('Profils par domaine (doit être ~équilibré) :')
for code_, n in sorted(label_counts.items()):
    print(f'  {code_}: {n}')

n_noisy = sum(1 for r in rows if r['label_noise'])
print(f"\nProfils à étiquette bruitée : {n_noisy} ({n_noisy/len(rows):.1%})")

In [ ]:
matiere_tokens = Counter()
for r in rows:
    for tag in r['matieres_preferees'].split(', '):
        if tag:
            matiere_tokens[tag] += 1
matiere_tokens.most_common(10)

## 3. Préparation des features et séparation train/validation

Encodage multi-hot des matières/compétences/intérêts déclarés + one-hot environnement/série de bac (`ml.features.encode_batch`). Split 80/20 reproductible (même graine que `ml.train`).

In [ ]:
from ml.train import _train_val_split, _rows_to_xy, N_PER_CLASS, SEED, VAL_FRACTION

train_rows, val_rows = _train_val_split(rows, VAL_FRACTION, SEED)
X_train, y_train = _rows_to_xy(train_rows)
X_val, y_val = _rows_to_xy(val_rows)
X_train.shape, X_val.shape

## 4. Modèle de référence simple, puis comparaison de deux approches

- `NearestCentroidBaseline` : le modèle de référence exigé par le brief (section 7).
- `KNNClassifier` : k plus proches voisins (distance cosinus).
- `SoftmaxRegression` : régression logistique multinomiale (descente de gradient, numpy pur — voir `ml/models.py` pour la justification de l'implémentation from-scratch).

In [ ]:
from ml.models import NearestCentroidBaseline, KNNClassifier, SoftmaxRegression

baseline = NearestCentroidBaseline().fit(X_train, y_train)
knn = KNNClassifier(k=15).fit(X_train, y_train)
softmax = SoftmaxRegression(lr=0.5, l2=1e-3, epochs=400, seed=SEED).fit(X_train, y_train)
print('Modèles entraînés.')

## 5. Évaluation — au-delà de la simple accuracy

In [ ]:
from ml import metrics
from ml.models import predict_labels

classes = sorted(DOMAINE_IDS)
for name, model in [('baseline', baseline), ('knn', knn), ('softmax', softmax)]:
    proba = model.predict_proba(X_val)
    y_pred = predict_labels(model, X_val)
    print(f"{name:10s} acc={metrics.accuracy(y_val, y_pred):.3f}  "
          f"top3={metrics.top_k_accuracy(y_val, proba, classes, k=3):.3f}  "
          f"macroF1={metrics.macro_f1(y_val, y_pred, classes):.3f}  "
          f"MRR={metrics.mean_reciprocal_rank(y_val, proba, classes):.3f}  "
          f"stabilite={metrics.stability_score(model, X_val):.3f}")

In [ ]:
import numpy as np

y_pred_softmax = predict_labels(softmax, X_val)
cm = metrics.confusion_matrix(y_val, y_pred_softmax, classes)
print('Matrice de confusion (softmax), lignes=vrai, colonnes=prédit:')
print(np.array2string(cm, max_line_width=200))

## 6. Généralisation vers l'enquête réelle

Montage recommandé par le brief : entraînement sur synthétique, test de généralisation sur les réponses réelles collectées (`ORIENT'IA — Réponses.xlsx`). Voir `data/ml/survey/registre_collecte.md` pour la traçabilité complète de cette collecte, et la limite explicite sur sa taille.

In [ ]:
from ml.train import _evaluate_on_survey

survey_eval = _evaluate_on_survey(softmax, classes)
survey_eval

## 7. Artifacts produits

`python -m ml.train` régénère automatiquement :
- `ml/artifacts/model.json` — modèle sélectionné (softmax), rechargé par `ml/inference.py` pour les outils de l'agent conversationnel ;
- `ml/artifacts/evaluation_results.json` / `evaluation_report.md` — le rapport complet (comparaison des 3 approches, calibration, biais, erreurs, généralisation) ;
- `data/ml/synthetic/` et `data/ml/survey/` — les jeux de données et leur documentation/registre de traçabilité.